In [34]:
import pulp
import numpy as np

In [ ]:
v_out_target = 3
r_top_min = 90 * 1e3
available_resistors_kOm = np.array([22, 22.1, 23.2, 26.1, 30, 33, 39.2, 44.2, 51, 56, 62, 71.5, 75, 86.6, 100, 124, 137, 178, 200, 226, 249, 300, 536, 680])

v_fb = 0.6

In [36]:
available_resistors_Om = available_resistors_kOm * 1e3

In [37]:
model = pulp.LpProblem("NearestVoltageDivider", pulp.LpMinimize)

In [38]:
r_h = pulp.LpVariable("r_h", lowBound=0, cat="Integer")
r_l = pulp.LpVariable("r_l", lowBound=0, cat="Integer")

In [39]:
up_resistor_used = pulp.LpVariable.dicts("up_vars", available_resistors_Om, cat="Binary")
down_resistor_used = pulp.LpVariable.dicts("down_vars", available_resistors_Om, cat="Binary")

In [40]:
model += pulp.lpSum([down_resistor_used[val] for val in available_resistors_Om]) == 1
model += pulp.lpSum([up_resistor_used[val] for val in available_resistors_Om]) == 1

In [41]:
r_h_expr = pulp.lpSum([r * up_resistor_used[r] for r in available_resistors_Om])
r_l_expr = pulp.lpSum([r * down_resistor_used[r] for r in available_resistors_Om])

In [42]:

error = pulp.LpVariable("error", lowBound=0)
model += v_out_target * r_l_expr - v_fb * (r_h_expr + r_l_expr) <= error
model += v_out_target * r_l_expr - v_fb * (r_h_expr + r_l_expr) >= -error
model += error

model += r_h_expr >= r_top_min

In [43]:
model.solve()

print("State:", pulp.LpStatus[model.status])

r_h_solved = sum(r * pulp.value(up_resistor_used[r]) for r in available_resistors_Om)
r_l_solved = sum(r * pulp.value(down_resistor_used[r]) for r in available_resistors_Om)

State: Optimal


In [44]:
v_out_solved = ((r_h_solved / r_l_solved) + 1) * v_fb
print(f"Calculated resistors: R_H: {r_h_solved} Om; R_L: {r_l_solved} Om;")
print(f"Solved voltage: {v_out_solved} V")
print(f"Voltage error(abs(target - solved)): {abs(v_out_target - v_out_solved)} V")


Calculated resistors: R_H: 249000.0 Om; R_L: 62000.0 Om;
Solved voltage: 3.009677419354839 V
Voltage error(abs(target - solved)): 0.00967741935483879 V
